# Phase 15: YOLOv8l High-Resolution Detection (Sanity Run)

This notebook tests the YOLOv8l (Large) architecture natively at `1024x1024` on a Tesla T4 to ensure we don't OOM. We are starting with `batch=4` for 3 epochs.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install dependencies
!pip install ultralytics


In [3]:
import os
import torch
from ultralytics import YOLO

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


Current working directory: /content/drive/MyDrive/sem_defect_project
CUDA Available: True
GPU Device: Tesla T4


In [4]:
# Verify baseline weights exist
weights_path = 'yolov8l.pt'
if not os.path.exists(weights_path):
    print("Downloading baseline yolov8l.pt...")
    YOLO(weights_path)


## 1. Sanity Run (Batch=4, Epochs=3)

In [5]:
import time

# Load official COCO-pretrained YOLOv8l
model = YOLO('yolov8l.pt')

print("Starting Sanity Run (Batch=4)...")
t0 = time.time()

# Train for 3 epochs to test VRAM
results = model.train(
    data='dataset_yolo_single_class/data.yaml',
    epochs=3,
    imgsz=1024,          # High-Res Target
    batch=4,             # Testing batch=4 for 15GB VRAM
    name='EXP15-Sanity-v8l-1024',
    cache=False,         # Disable RAM caching
    amp=True,            # Enable FP16 Mixed Precision
    exist_ok=True
)

t1 = time.time()
print(f"\nSanity Run Complete in {t1 - t0:.2f} seconds.")


Starting Sanity Run (Batch=4)...
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EXP15-Sani

In [6]:
# Print Peak VRAM Usage
print("\n=== TESLA T4 VRAM USAGE ===")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv



=== TESLA T4 VRAM USAGE ===
memory.used [MiB], memory.total [MiB]
2553 MiB, 15360 MiB
